# Cyclistic Membership Conversion Analysis

**Google Data Analytics Capstone - portfolio edition**  
**Business question:** How do annual members and casual riders use Cyclistic bikes differently, and how can those differences inform membership-conversion marketing?

This notebook follows the **Ask -> Prepare -> Process -> Analyze -> Share -> Act** framework. The raw Divvy files are intentionally not redistributed. The repository includes reproducible cleaning code in `src/analysis_pipeline.py` and precomputed aggregate tables in `outputs/summary_tables/`.

## 1. Ask

The marketing team wants evidence that can support campaign decisions. The analytical task is therefore not merely to describe rides, but to identify **behavioral contexts where casual riders differ materially from members** and turn those contexts into testable marketing hypotheses.

Primary dimensions: ride duration, day/weekend mix, hour-of-day pattern, seasonality, same-station behavior, stations, and routes.

## 2. Prepare

The supplied archive contains many years but has gaps in newer monthly data. **2019 is the most recent complete uninterrupted 12-month window in the supplied files**, allowing a full seasonal cycle.

The four quarterly files contain 3.82M rides. Q2 uses verbose column names while Q1/Q3/Q4 use the legacy short names. The pipeline harmonizes them and maps `Subscriber -> member`, `Customer -> casual`.

Gender and birth-year fields are excluded because they are unnecessary for the assigned business question.

In [1]:
from pathlib import Path
import pandas as pd
import json

ROOT = Path('..') if Path.cwd().name == 'notebooks' else Path('.')
TABLES = ROOT / 'outputs' / 'summary_tables'
FIGURES = ROOT / 'outputs' / 'figures'

audit = json.load(open(TABLES / 'audit_metrics.json'))
audit

{'raw_rows': 3818004,
 'invalid_datetime_rows': 0,
 'valid_positive_duration_rows': 3818004,
 'analysis_rows': 3818004,
 'rides_under_1_min': 0,
 'rides_over_2_hours': 42539,
 'rides_over_24_hours': 1849,
 'missing_start_station_rows': 0,
 'missing_end_station_rows': 0,
 'missing_user_type_rows': 0,
 'unmapped_user_type_rows': 0,
 'source_duration_missing_rows': 0,
 'source_duration_discrepancy_gt2sec_rows': 94,
 'duplicate_ride_id_hashes': 0,
 'member_rides': 2937367,
 'member_share': 0.7693462343151029,
 'member_median_duration_min': 9.800000190734863,
 'member_mean_duration_min': 14.327797201932446,
 'member_weekend_share': 0.184996971777786,
 'member_same_station_share': 0.016221670632236285,
 'member_summer_share': 0.384751037238452,
 'member_peak_hour': 17,
 'member_peak_weekday': 'Tuesday',
 'member_peak_month': 'August',
 'member_weekday_commute_window_share_all_rides': 0.4763882075341624,
 'member_weekday_commute_window_share_weekday_rides': 0.5845232361569498,
 'member_weeken

## 3. Process

Key quality decisions:

- No duplicate ride IDs were detected.
- No rider-type or station-name values are missing in the core 2019 files.
- Thirteen rides on **3 November 2019** produce a non-positive timestamp difference because the local clock moved backward for daylight-saving time. The pipeline preserves these valid rides using the positive source `tripduration` field.
- Extreme rides longer than 24 hours are retained in volume/station counts but flagged for duration sensitivity. Median ride duration is the main duration KPI because the distribution is strongly right-skewed.

In [2]:
quality = pd.read_csv(TABLES / 'data_quality_audit.csv')
quality

,check,count,treatment
0,Raw trip records,3818004,Four quarterly 2019 Divvy trip files
1,Duplicate ride IDs,0,None detected
2,Invalid timestamps,0,None detected
3,Calculated durations <= 0,13,13 daylight-saving fallback cases; recovered u...
4,Duration fallback to source field,13,Used for DST-affected rides
5,Missing start station names,0,None detected
6,Missing end station names,0,None detected
7,Missing rider type,0,None detected
8,Unmapped rider type,0,None detected
9,Rides > 24 hours,1849,Retained for volume counts; flagged as duratio...


## 4. Analyze - rider mix and duration

In [3]:
membership = pd.read_csv(TABLES / 'membership_mix.csv')
duration = pd.read_csv(TABLES / 'duration_summary.csv')

membership, duration[['member_casual','rides','mean_duration_min','median_duration_min','p90_duration_min','p99_duration_min']]

(  member_casual    rides     share
 0        casual   880637  0.230654
 1        member  2937367  0.769346,
   member_casual    rides  ...  p90_duration_min  p99_duration_min
 0        casual   880637  ...         81.400002        203.143995
 1        member  2937367  ...         24.900000         44.616665
 
 [2 rows x 6 columns])

**Interpretation:** Members generate most rides, but casual trips are much longer. Median duration is the stronger business KPI because rare multi-day rides inflate means.

![Median ride duration](../outputs/figures/02_median_duration.png)

## 5. Analyze - weekday/weekend and hour-of-day behavior

In [4]:
weekday = pd.read_csv(TABLES / 'weekday_usage.csv')
day_type = pd.read_csv(TABLES / 'day_type_usage.csv')
hourly = pd.read_csv(TABLES / 'hourly_usage.csv')

day_type

,member_casual,day_type,rides,share
0,casual,Weekday,502402,0.570498
1,casual,Weekend,378235,0.429502
2,member,Weekday,2393963,0.815003
3,member,Weekend,543404,0.184997


Casual riders are much more weekend-oriented. Member activity is more concentrated in weekday 7-9am and 4-6pm windows, a pattern **consistent with** recurring commute use. The data does not contain trip purpose, so the analysis avoids claiming that all such rides are commutes.

![Hourly profile](../outputs/figures/05_hourly_profile.png)

## 6. Analyze - seasonality

In [5]:
monthly = pd.read_csv(TABLES / 'monthly_usage.csv')
monthly.pivot(index=['month','month_name'], columns='member_casual', values='rides')

,member_casual,casual,member
month,month_name,,
1,January,4602,98670
2,February,2638,93548
3,March,15923,149688
4,April,47744,217566
5,May,81624,285834
6,June,130218,345177
7,July,175632,381683
8,August,186889,403295
9,September,129173,364046


![Monthly ride volume](../outputs/figures/04_monthly_usage.png)

## 7. Analyze - same-station behavior and location opportunities

In [6]:
same_station = pd.read_csv(TABLES / 'same_station_usage.csv')
hotspots = pd.read_csv(TABLES / 'high_casual_share_stations.csv')

same_station, hotspots.head(10)

(  member_casual  same_station_rides    rides  same_station_share
 0        casual              104598   880637            0.118775
 1        member               47649  2937367            0.016222,
           start_station_name  member_rides  ...  total_rides  casual_share
 0  Lake Shore Dr & Monroe St       10566.0  ...      49804.0      0.787848
 1    Streeter Dr & Grand Ave       14879.0  ...      67983.0      0.781136
 2             Shedd Aquarium        5815.0  ...      26432.0      0.780002
 3               Field Museum        2402.0  ...      10056.0      0.761138
 4             Dusable Harbor        4607.0  ...      17153.0      0.731417
 5          Adler Planetarium        4807.0  ...      16735.0      0.712758
 6            McCormick Place        1931.0  ...       5628.0      0.656894
 7      Michigan Ave & 8th St        5269.0  ...      14794.0      0.643842
 8            Millennium Park       12357.0  ...      34106.0      0.637688
 9      Michigan Ave & Oak St       14061

Same-station trips are a particularly strong separator between segments and are consistent with round-trip/leisure use. High-volume stations with very high casual shares provide concrete locations for conversion experiments.

![High casual-share stations](../outputs/figures/07_casual_hotspots.png)

## 8. Share - executive dashboard

![Executive dashboard](../outputs/figures/09_executive_dashboard.png)

## 9. Act - recommendations

1. **Geo-target high-casual stations.** Run localized conversion tests at high-volume stations where casual share is already high.
2. **Time campaigns around casual demand.** Emphasize weekends, warm months, and midday/afternoon casual periods; test a separate treatment for weekday commute-window casual demand.
3. **Segment the value proposition and test it.** Leisure/round-trip contexts should receive different creative from recurring weekday contexts. Use privacy-safe first-party measurement and A/B tests to establish which treatments actually increase membership.

### Next analytical step
Refresh the pipeline on a complete recent year and connect campaign exposure to conversion outcomes. Trip data alone cannot identify repeated casual riders or prove what causes a rider to buy a membership.

## Reproducibility

To rebuild the aggregate tables from raw files:

```bash
python src/analysis_pipeline.py --raw-dir data/raw --output-dir outputs/summary_tables
```

See `README.md`, `docs/methodology.md`, and `sql/analysis_queries.sql` for the complete portfolio narrative and technical layer.